# 膝MRIの12所見を当てる — データを見てからベースラインを決めるまで

RSNA Knee Abnormality Detection は、膝のMRI検査1件について
**12種類の所見それぞれが「ある/なし」の確率**を出すコンペです。
スコアは12所見の AUC の平均(macro AUC)。

このノートブックは、**データを見て分かったこと**と、
**そこから決まったベースラインの構成**を順に書きます。
モデルを凝る前に決めておくべきことが、このコンペにはいくつもあります。

本ノートブックでお伝えすること:

- このコンペの本当の難所はどこか(画像処理ではありません)
- データを見て分かる「そのまま使ってはいけない列」
- 検証(CV)をどう切るべきか
- 上の全部から、ベースラインの設定がどう決まるか

最後に、実際に提出して確かめた **public LB 0.898 までの道筋**を、
段階的に説明します。

## 0. まずコンペの形を押さえる

| | |
|---|---|
| 入力 | 1件の検査(study)= 複数の撮像シリーズ(矢状断・冠状断・軸位断など) |
| 出力 | 12所見の確率 |
| 評価 | 12所見の AUC の平均 |
| 形式 | Code Competition(ノートブックを提出。実行9時間以内) |

12所見はこれです。

`ACL`(前十字靭帯) `MCL`(内側側副靭帯) `Medial Meniscus`(内側半月板)
`Lateral Meniscus`(外側半月板) `Medial OA` `Lateral OA` `PF OA`(3か所の変形性関節症)
`Effusion`(関節液貯留) `Synovitis`(滑膜炎) `Baker's`(ベーカー嚢腫)
`Contusion`(骨挫傷) `Fracture`(骨折)

1件の検査(study)に何が入っていて、何を当てるのかを図にすると、こうなります。

```
【 1件の検査 (study) 】

  画像 ─┬─ 矢状断 (Sagittal) のシリーズ ─┐
        ├─ 冠状断 (Coronal)  のシリーズ ─┼─ 各シリーズ = 数十枚のスライス
        └─ 軸位断 (Axial)    のシリーズ ─┘   (膝を輪切りにした断面画像)

  文章 ─── 放射線科医が書いた所見レポート 1本
           (英語とは限らない。9言語が混在 → §2)

                          │
                          ↓  予測する

  ┌──────────────────────────────────────────┐
  │  12所見それぞれの「ある/なし」の確率        │
  │  ACL / MCL / Medial Meniscus / ...        │
  └──────────────────────────────────────────┘
```

同じ膝を3つの向きから撮っているのがポイントです。
どの所見がどの向きで見えるかは決まっており、これは後で実測します(§6)。

医学用語が並びますが、中身はどれも「膝のどこが、どう傷んでいるか」です。
1つだけ例を挙げます。**`ACL`(前十字靭帯)は、膝の中央で太ももの骨とすねの骨を
つないでいる靭帯**で、スポーツ中の急な方向転換などで切れることがあります。
この所見は「その靭帯が切れているかどうか」を当てるものです。
残り11個も同じように、**部位(靭帯・半月板・軟骨・骨・関節液)と
傷み方の組み合わせ**になっています。
所見の医学的な中身をこれ以上知らなくても、以降の話は追えます。

### 評価指標から決まること

AUC は **順位しか見ません**。「0.9 と 0.8」でも「0.6 と 0.5」でも、
順番が同じなら同じスコアです。ここから2つ決まります。

1. **確率を較正する必要がない。** 予測値を単調に変換してもスコアは変わりません
2. **モデルを混ぜるときは順位に直してから混ぜられる。**
   出力のスケールが違うモデル同士でも、順位にすれば公平に平均できます


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path("/kaggle/input/rsna-knee-abnormality-detection"),
                        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")]
            if p.exists())

train = pd.read_csv(ROOT / "train.csv")
series = pd.read_csv(ROOT / "train_series.csv")

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]

print(f"train   : {len(train):,} 件の検査")
print(f"series  : {len(series):,} 本のシリーズ")


## 1. 学習データの重要な前提 — 正解ラベルは 58 件しかない

`train.csv` を開くと、すぐに分かります。

**train には 4,407 件の検査がありますが、12所見のラベルが付いているのは 58 件だけです。**
残りの 4,349 件には、**放射線科医が書いた所見レポートの原文**だけが付いています。

```
train 4,407 件
  │
  ├─    58 件 : 画像 + レポート + 【12所見の正解ラベル】
  │              └─→ 正解があるので「作ったラベルの品質チェック」に使える
  │
  └─ 4,349 件 : 画像 + レポート + (ラベルなし)
                 │
                 └─→ このままでは学習に使えない
                     レポートを読んでラベルを作れば使えるようになる
```

つまり、4,349 件を学習に利用するためには、
**レポートの文章を読んで、そこから12所見の「ある/なし」を自分で決める**必要があります。
この作業をしない限り、学習に使えるデータは 58 件のままです。
58 件で画像モデルを学習するのは、現実的ではありません。

そのため、このコンペでは「画像から所見を当てる問題」に取りかかる前に、
**「レポートから学習用のラベルを作る問題」**を先に解くことになります。
手順にすると3段階です。

1. レポートを読んで、4,407 件ぶんのラベルを**自分で作る**
2. そのラベルで画像モデルを学習する
3. 58 件の正解ラベルは、**作ったラベルの品質チェック**に使う

ラベル作りが適切でないと、その先の画像モデルをいくら凝っても頭打ちになります。
実際、あとで示すように、**このコンペで一番スコアが動いたのはラベルを差し替えたとき**でした。


In [ ]:
# ---- ラベルが付いているのは何件か -------------------------------------------
labeled = train[TARGETS].notna().all(axis=1)
print(f"12所見すべてにラベルがある検査: {labeled.sum()} / {len(train):,} 件")
print(f"レポート本文がある検査        : {train['Report'].notna().sum():,} 件")

print("\n--- レポートの例(先頭200字)---")
print(train.loc[train["Report"].notna(), "Report"].iloc[0][:200])


## 2. レポートは9言語ある

レポートは英語だけではありません。英語・スペイン語・フランス語・オランダ語・
ドイツ語・トルコ語・クロアチア語・ギリシャ語・ロシア語が混ざっています。

ここでの設計判断は **「言語を判定してから処理を分ける」ことをしない**、です。

理由は単純で、**言語判定を間違えたらそのレポートは丸ごと読めなくなる**から。
「the が含まれるから英語」のような安い判定は、
オランダ語やドイツ語の文章でも普通に当たってしまいます。

代わりに、**9言語ぶんの手がかりを全部まとめた辞書**を作り、
どのレポートに対しても全部を当てにいきます。
言語を当てる必要がなくなるので、判定ミスという失敗の種が消えます。

読むときに必要なのは3種類の表現です。

| 種類 | 例 |
|---|---|
| 所見そのもの | `anterior cruciate`, `cruzado anterior`, `vorderes kreuzband`, `мениск` |
| 否定 | `no`, `sin`, `ohne`, `unremarkable` |
| ぼかし | `possible`, `suspected`, `cannot exclude` |

否定とぼかしを拾わないと、「ACL断裂はありません」を「ACL断裂あり」と読んでしまいます。


In [ ]:
# ---- 何語で書かれているか ---------------------------------------------------
# 言語判定はしない(§2)。ここでは「英語以外が混ざっている」ことの確認だけ。
CUE = {
    "英語":       r"\bmeniscus\b|\bligament\b",
    "スペイン語": r"\bmenisco\b|\bligamento\b",
    "フランス語": r"\bm[eé]nisque\b|ligament crois",
    "オランダ語": r"\bkruisband\b",
    "ドイツ語":   r"\bmeniskus\b|\bkreuzband\b",
    "トルコ語":   r"men[iı]sk[uü]s|çapraz bağ",
    "ロシア語":   r"мениск|связк",
    "ギリシャ語": r"μηνίσκ|σύνδεσμ",
}
rep = train["Report"].fillna("").str.lower()
for name, pat in CUE.items():
    n = rep.str.contains(pat, regex=True).sum()
    print(f"  {name:8s} を示す語を含む: {n:5,} 件")
print("\n→ 英語だけではない。言語判定で分岐せず、全言語の辞書を一度に当てる(§2)")


## 3. 画像データの実態を数字で見る

学習用の画像データ量は下記のとおりです。

| | |
|---|---|
| DICOM ファイル数 | **819,078** |
| 合計サイズ | **約 490 GB** |
| ファイルサイズの中央値 | 0.526 MB |

中央値 0.526 MB は、512×512 の16bit 画像を**無圧縮**で保存したときの
サイズとぴったり一致します。実際、サンプルした 22,859 ファイルは
すべて非圧縮(Explicit VR Little Endian)でした。

**分かること**: 復号は速い(圧縮を解く処理がない)。重いのは
「大量のファイルを開くこと」と「リサイズ」の方です。

**やること**: 490 GB を毎エポック読み直すのは論外なので、
**前処理を一度だけ実行して小さな配列にキャッシュ**します。
学習はキャッシュだけを読みます。

## 4. 用意されている列が何を表しているかを確認する

`train_series.csv` には各シリーズの説明として
`Anatomical_Plane`(撮像面)、`Fluid_Sensitive`、`Fat_Suppression` が付いています。

名前からは `Fluid_Sensitive`(水を強調する撮り方か)と
`Fat_Suppression`(脂肪を消す撮り方か)は**別の性質**に見えます。
物理的にも独立な性質です。ところが、中身を数えると:

**この2列は 24,371 本すべてで完全に一致していました(100%)。**
片方は、もう片方の複製です。**2列あるように見えて、情報は1つ分しかありません。**

さらに、その1列が何を表しているのかも調べました。
`Fluid_Sensitive` の値ごとに、実際のシーケンス種別(T1・PD・T2 という
3つの撮り方)を数えるとこうなります。

```
Fluid_Sensitive = 0 の中身 : T1 4,219 / PD 2,361 / T2 1,782 / 不明 1,999
Fluid_Sensitive = 1 の中身 : PD 7,712 / T2 2,699 / T1     4 / 不明 3,595
```

T1・PD・T2 は**まったく写り方が違う撮り方**です。
値が 0 のシリーズには T1 が 4,219 本・PD が 2,361 本・T2 が 1,782 本と3種類が混在し、
値が 1 のシリーズにも PD 7,712 本と T2 2,699 本が混在しています。
**したがって `Fluid_Sensitive` の値を見ても、そのシリーズが T1・PD・T2 の
どれで撮られたのかは判別できません。**
この列から読み取れるのは脂肪抑制の有無だけで、T1/PD/T2 の区別は含まれていません。

まとめると、与えられた2列から取り出せる情報は「脂肪抑制の有無」だけ。
T1/PD/T2 の区別は**自分で復元する必要があります**。

### 撮り方の種別は DICOM ヘッダから復元できる

MRI は、装置が体に電波を当てて、返ってくる信号を画像にします。
このとき **電波を当てる間隔**(`RepetitionTime` / TR)と、
**当ててから信号を読み取るまでの待ち時間**(`EchoTime` / TE)を変えると、
同じ膝でも「何が白く写るか」が変わります。
T1・PD・T2 という種別は、もともとこの2つの数値の組み合わせで決まるものなので、
DICOM ヘッダに残っている TR と TE から逆算できます。

| 条件 | 種別 | 何が見やすいか |
|---|---|---|
| TR < 800 | T1 | 脂肪が白い。解剖の形 |
| TE < 50 | PD | 中間。半月板 |
| それ以外 | T2 | 水が白い。関節液・炎症 |

シリーズ名のテキストからも読めますが、**11.8% は匿名化されて
`DummySeriesDesc!` になっている**ので、テキストだけでは 77% しか分類できません。
TE/TR は 95.1% のシリーズに存在するので、**両方使えば 100% 分類できます**。

> この 11.8% / 95.1% は**全 24,371 シリーズのヘッダを1度走査した実測値**です。
> 下のセルは実行時間を抑えるため 300 シリーズの標本で確かめているので、
> 表示される割合はこの値から多少ずれます。

> 用意された列は、名前が示す内容と中身が一致しているとは限りません。
> ここでは2列を突き合わせて数えたことで、
> **片方がもう片方の複製であること**と、
> **T1/PD/T2 の区別がどちらの列にも入っていないこと**の2点が分かりました。
> 数えずに列名のまま特徴量として使っていたら、
> この2点に気づかないまま学習を進めることになります。


In [ ]:
# ---- 用意された列は「別の性質」を表しているか(§4)-------------------------
ct = pd.crosstab(series["Fluid_Sensitive"], series["Fat_Suppression"])
print("Fluid_Sensitive x Fat_Suppression:")
print(ct)

agree = (series["Fluid_Sensitive"] == series["Fat_Suppression"]).mean()
print(f"\n2列が一致する割合: {agree:.1%}")
print("→ ほぼ同じことを言っている。コントラスト(T1/PD/T2)の情報は入っていない")
print("   復元するには DICOM の TR/TE を見る(§4)")


In [ ]:
# ---- DICOM ヘッダを覗く(物理スケールとコントラスト)------------------------
# 全 819,078 ファイルを開くのは高いので、シリーズごとに 1 枚だけ、
# ランダムな 300 シリーズをサンプルする。傾向を見るにはこれで十分。
import pydicom

sample = series.sample(min(300, len(series)), random_state=0)

rows = []
for r in sample.itertuples():
    d = ROOT / "train_series" / r.StudyInstanceUID / r.SeriesInstanceUID
    files = list(d.glob("*.dcm"))[:1] if d.exists() else []
    if not files:
        continue
    try:
        ds = pydicom.dcmread(files[0], stop_before_pixels=True)
    except Exception:
        continue
    ps = getattr(ds, "PixelSpacing", None)
    rows.append(dict(
        spacing=float(ps[0]) if ps is not None else np.nan,
        rows_px=float(getattr(ds, "Rows", np.nan)),
        TR=float(getattr(ds, "RepetitionTime", np.nan) or np.nan),
        TE=float(getattr(ds, "EchoTime", np.nan) or np.nan),
        desc=str(getattr(ds, "SeriesDescription", "")),
    ))

hdr = pd.DataFrame(rows)
print(f"読めたシリーズ: {len(hdr)}")
print("\n1ピクセルが表す長さ (mm):")
print(hdr["spacing"].describe()[["min", "25%", "50%", "75%", "max"]].round(3))
print("\n→ 施設・装置でばらつく。リサイズだけでは大きさが揃わない(§5)")


In [ ]:
# ---- TR/TE からコントラストを復元する(§4)---------------------------------
def contrast_of(tr, te):
    """TR/TE の組から T1 / PD / T2 を決める。単純な規則で十分に効く。"""
    if not np.isfinite(tr) or not np.isfinite(te):
        return "不明"
    if tr < 800:
        return "T1"
    if te < 50:
        return "PD"
    return "T2"


hdr["contrast"] = [contrast_of(t, e) for t, e in zip(hdr["TR"], hdr["TE"])]
print("TR/TE から決めたコントラスト:")
print(hdr["contrast"].value_counts().to_string())

dummy = hdr["desc"].str.contains("Dummy", case=False, na=False).mean()
print(f"\nシリーズ名が匿名化されている割合: {dummy:.1%}")
print("→ 名前だけに頼ると分類できない。TR/TE と併用する(§4)")


## 5. 物理スケールを揃える — 160mm では 28.9% で効いていなかった

MRI の画像は、施設や装置によって**1ピクセルが表す実際の長さが違います**。
DICOM の `PixelSpacing` に mm/ピクセルが入っています。

これを無視して「全部 224×224 にリサイズ」すると、
**同じ半月板が画像によって2倍以上の大きさで写る**ことになります。
モデルにとっては別物です。

そこで、**膝関節が収まる物理的な視野(mm)を決めて、そこを切り出してからリサイズ**します。

```
切り出す幅(ピクセル) = 視野(mm) ÷ PixelSpacing(mm/ピクセル)
```

最初は視野を 160mm にしました。ところが**うまくいっていませんでした**。

撮像された画像そのものが 160mm より狭いことがあり、
そのときは切り出し幅が画像サイズで頭打ちになります。
**このとき、物理スケールを揃える処理は実質的に働いていません**
(画像全体をそのまま使っているのと同じ状態になります)。

数えたところ、**160mm では 28.9% のシリーズでこれが起きていました**。
視野を 130mm にすると 0.5% まで下がります(こちらも全シリーズの走査による値。
下のセルは 300 シリーズの標本なので数字は少しずれます)。

この1点だけを直した提出で、LB は 0.857 → 0.861 になりました。

> **教訓**: 「実装した」と「効いている」は違います。
> 効いているかは、**効かなかった件数を数えて**確かめます。


In [ ]:
# ---- 視野を何 mm にすべきか(§5)-------------------------------------------
# 切り出し幅 = 視野(mm) / spacing。これが画像サイズを超えると頭打ちになり、
# 物理スケール合わせが働かなくなる。その割合を視野ごとに数える。
ok = hdr.dropna(subset=["spacing", "rows_px"])
print(f"{'視野':>6s}  {'働かないシリーズ':>18s}")
for fov in (160, 150, 140, 130, 120):
    need = fov / ok["spacing"]
    clipped = (need > ok["rows_px"]).mean()
    print(f"{fov:4d}mm  {clipped:17.1%}")
print("\n→ 160mm は無視できない割合で働かない。130mm を採用した(§5)")


## 6. どの撮像面に、どの所見が写るのか

膝のMRIは複数の向きから撮ります。矢状断(S)・冠状断(C)・軸位断(A)。

「全部入れればよい」と決める前に、**平面を減らした版を作って提出**し、
どの所見がどの平面で決まるのかを実測しました。

| 使った平面 | LB |
|---|---|
| 冠状のみ (C) | 0.718 |
| 矢状のみ (S) | 0.772 |
| 矢状 + 軸位 (S+A) | 0.802 |
| **3平面すべて** | **0.820** |

所見ごとに見ると、解剖学的に納得できる並びになりました。
以下は**手元の検証スコア(所見ごとの AUC)**で、上の表の LB とは別物です。

| 所見 | 最も効く単一平面(検証AUC) | 解剖学的な期待 |
|---|---|---|
| MCL(内側側副靭帯) | **冠状 0.705**(矢状は 0.477 と最低) | 内側側副靭帯は冠状断で見る ✓ |
| Medial Meniscus | **矢状 0.729** | 半月板は矢状断 ✓ |
| PF OA(膝蓋大腿) | **軸位 0.629**(矢状 0.590 / 冠状 0.528) | 膝蓋骨の裏の軟骨は軸位 ✓ |
| Synovitis | **軸位 0.804** | 関節液・滑膜は軸位 ✓ |

**モデルが解剖学的に妥当なところを見ている傍証**になります。
おかしなショートカット(たとえば施設ごとの画像のクセ)を
学習しているわけではなさそうだ、と言えます。

2枚目に足すなら矢状+軸位が良い、というのも検証スコアで確認できました
(矢状+軸位 0.714 / 矢状+冠状 0.694)。

## 7. 検証(CV)の切り方 — テストと同じ切れ方を再現する

CV の目的は、**提出したときのスコアを手元で予測すること**です。
なので、**テストデータと同じ切れ方**で切る必要があります。

ランダムに5分割すると何が起きるか。
同じ施設・同じ装置で撮られた似た検査が、学習側と検証側の**両方に入ります**。
モデルは「この装置の画像はこう写る」を覚えるだけで検証スコアを上げられてしまい、
**手元では良いのに提出すると落ちる**ことになります。

そこで、**装置を識別する複合キー**(以下「装置の指紋」と呼びます)を作り、
その単位で分けます。

```
指紋 = Manufacturer | ManufacturerModelName | SoftwareVersions | MagneticFieldStrength
```

4,407 件はこの指紋で **112 群**に分かれます。
群がまたがらないように5分割すれば、
検証側は「学習で見たことのない装置」になります。

### 指紋に使えなかった列

最初は `ImagingFrequency`(撮像周波数)も指紋に入れようとしました。
数えたら **8,023 種類**あり、**4,407 件のうち 3,978 件では
1つの検査の中で値が複数**ありました。

これは装置のIDではなく、**撮像のたびに再調整される値**です。
入れると群がバラバラになり、グループ分割の意味がなくなります。

> **教訓**: グループのキーは「同じものには同じ値が入るか」を数えて確かめます。
> 名前から選ぶと外します。


In [ ]:
# ---- 検証の切り方(§7)------------------------------------------------------
# 装置の指紋でグループを作り、群がまたがらないように5分割する。
# 指紋は全シリーズのヘッダを1度だけ走査して作る(ここでは分割の作法だけ示す)。
from sklearn.model_selection import GroupKFold

demo = train[["StudyInstanceUID"]].copy()
demo["group"] = np.random.default_rng(0).integers(0, 112, len(demo))  # 実データでは112群

gkf = GroupKFold(n_splits=5)
for f, (tr_i, va_i) in enumerate(gkf.split(demo, groups=demo["group"])):
    a = set(demo.iloc[tr_i]["group"])
    b = set(demo.iloc[va_i]["group"])
    assert not (a & b), "群が学習側と検証側にまたがっている"
    print(f"  fold {f}: 学習 {len(tr_i):,} / 検証 {len(va_i):,}  群の重なり {len(a & b)}")
print("\n→ 検証側は『学習で見たことのない装置』になる(§7)")


## 8. ここまでで、ベースラインの構成が決まる

EDA で分かったことが、そのまま設定になります。

| 決めたこと | 根拠(前の節) |
|---|---|
| **学習ターゲット = レポートから作ったラベル** | ラベルが 58 件しかない(§1) |
| **切り出し 130mm の物理スケール固定** | 160mm では 28.9% で働かない(§5) |
| **3平面すべてを入力に使う** | 平面ごとに得意な所見が違う(§6) |
| **スキャナ指紋の GroupKFold(5分割)** | 装置の学習を防ぐ(§7) |
| **合成は fold 平均、混ぜるときは順位平均** | AUC は順位しか見ない(§0) |

残るのは画像モデルの部分です。ここは一般的な構成で作ります。

### 入力の作り方(2.5D)

MRIは3次元ですが、3D CNN は重く、学習データも多くありません。
そこで **2.5D** にします。

1. 各平面から代表シリーズを1本選ぶ
2. スライスを等間隔に **24枚** サンプルする
3. **連続3枚を1組**にして、RGB の3チャンネルとして扱う(= 8組)
4. 3平面 × 8組 = 24枚の「画像」を、1件の検査として扱う

図にすると、1つの平面についてこうなります。

```
元のシリーズ(数十枚のスライス)
  ▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫▫

        ↓ 等間隔に 24 枚えらぶ

  ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪ ▪     24 枚
  1 2 3 4 5 6 7 8 ...                        24

        ↓ 連続する3枚を1組にまとめ、RGB の3チャンネルに入れる

  [1|2|3] [4|5|6] [7|8|9] ...  [22|23|24]      8 組
   R G B   R G B   R G B        R  G  B
   └─ 1枚のカラー画像のように扱える ─┘
```

3平面ぶんを束ねたものが、1件の検査の入力です。

```
  矢状断 ─→ 8組 ┐
  冠状断 ─→ 8組 ├─→ エンコーダに通す ─→ 24組ぶんの特徴ベクトル
  軸位断 ─→ 8組 ┘                              │
                                                ↓
                       所見ごとに「どの組を重視するか」を学習(§8 プーリング)
                                                ↓
                                        12所見それぞれの確率
```

隣り合うスライスをチャンネルに入れることで、
**平面に垂直な方向のつながり**を、2D のモデルのまま扱えます。
半月板のように「ある断面にだけ写る」構造は、その断面を含む組が強く重み付けされます。

### モデル

- **エンコーダ**: DINOv2 ViT-S/14(自己教師あり学習済み)
- **プーリング**: 8組のどれを重視するかを、**所見ごとに**学習する
  (半月板を読むのに良い位置と、ACL を読むのに良い位置は違うため)
- **学習率**: head は 1e-3、エンコーダは 8e-6 で**出力側6ブロックだけ**動かす

最後の点が効きます。事前学習済みのエンコーダを普通の学習率で全層動かすと、
**事前学習で獲得された重みが大きく書き換わり、その表現が失われます**。
実際、全層を 1e-4 で動かした版は、はるかに小さい resnet18 に負けました。
「適応させる、作り直さない」が正解でした。

## 9. どの変更が、どれだけ効いたか

上の構成は、いきなり全部決めたわけではありません。
変更を1つずつ切り分けて提出し、それぞれの寄与を測りました。

| 変更した点 | public LB |
|---|---|
| resnet18 2.5D + 自前ルールで作った弱ラベル | 0.820 |
| **ターゲットを、レポートを読ませたLLMのラベルに差し替え** | **0.857** (+0.037) |
| 切り出しを 160mm → 130mm(§5) | 0.861 (+0.004) |
| エンコーダを DINOv2 ViT-S/14 + 判別的な学習率へ | 0.882 (+0.021) |
| プーリングを所見ごとに(§8) | 0.884 (+0.002) |
| 入力解像度 224px → **336px** | **0.898** (+0.014) |

**一番大きく動いたのはラベルの差し替え(+0.037)** です。
画像モデルの工夫を全部足したより大きい。§1 で書いた
「これはラベルを作る問題である」が、そのまま結果に出ています。

解像度を上げたのが2番目(+0.014)。130mm を 336px で見ると
**1ピクセル 0.387mm** になります。半月板の裂け目を見るには、
このくらいの細かさが要る、ということだと考えています。

このように切り分けたので、**どの要素がどれだけ効いたのかを数字で言えます**。
まとめて変えていたら、伸びた分がラベルによるものか解像度によるものか分からず、
次にどこへ投資すべきかも決められませんでした。

## 10. ここから先

ベースラインができたら、次に効きそうなのはこのあたりです。

- **ラベルの質を上げる。** 一番動いた軸がここでした。
  レポートの読み取りを改善すれば、まだ伸びる余地があります
- **モデルを複数作って混ぜる。** ただし
  「単体で弱いモデルを混ぜても効かない」ことは実測しました。
  混ぜて効くのは**基準と同じくらい強くて、予測の傾向が違う**モデルだけです
- **公開されているモデルと混ぜる。** 他の人が別の設計で学習したモデルは、
  自分の中で工夫して作った多様性より、ずっと予測の傾向が違います


## 付録: このベースラインを実際に走らせる

ここまでは分析でした。以下のセルは §8〜§9 で説明した構成を実際に動かし、
`submission.csv` を書き出します。説明するだけでなく、**public LB 0.898 の提出物**
がこのノートブックから出ます。

重みは その構成の 5 fold を CC0 で公開したものです:
`kitopl/rsna-knee-eda-baseline-weights`。
学習に使ったラベルは公開の `stevenleehans/rsna-knee-llm-report-labels` なので、
非公開のデータには一切依存していません。

間違えやすい点が2つあります。

- **推論側の前処理は学習側と完全に一致していなければなりません。** 130mm の切り出し、
  24枚のスライス抽出、1〜99パーセンタイルでの正規化 — どれかがズレるとスコアが落ち、
  しかも何も教えてくれません。以下は学習時の前処理の逐語コピーです。
- **エンコーダは `pretrained=False` で組みます。** backbone の重みはチェックポイントに
  丸ごと入っているので、インターネット接続は要りません。


In [ ]:
# ---- 推論の設定 -------------------------------------------------------------
import os
import time
from concurrent.futures import ProcessPoolExecutor

import cv2
import pydicom
import timm
import torch
import torch.nn as nn
from scipy.stats import rankdata

OUT = "/kaggle/working"
INPUT = str(ROOT)

PLANES_ORDER = ["Sagittal", "Coronal", "Axial"]
PLANES, N_SLICES, HW = 3, 24, 336
TRIPLETS = N_SLICES // 3
FOV_MM = 130.0          # 学習時と揃える(§5)
BACKBONE = "vit_small_patch14_dinov2.lvd142m"
BATCH = 8
TIME_BUDGET = 7.5 * 3600   # 9時間制限に対する安全域。超えたら残りは0.5で埋める


def find_weights(fname="model_fold0.pt"):
    """重みの場所を /kaggle/input から探す。

    マウント先を決め打ちすると、データセットの付き方に依存して壊れる
    (実際に `/kaggle/input/<slug>` だと思って外した。正しくは
    `/kaggle/input/datasets/<owner>/<slug>`)。探して見つけ、
    見つからなかったときに原因が分かるよう中身も出す。
    """
    for root, _, files in os.walk("/kaggle/input"):
        if fname in files:
            return root
    return None


dev = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={dev}  torch={torch.__version__}  timm={timm.__version__}")
print("/kaggle/input の直下:", sorted(os.listdir("/kaggle/input")))
WEIGHTS = find_weights()
print("重みのディレクトリ:", WEIGHTS)
assert WEIGHTS, "model_fold0.pt が /kaggle/input のどこにも無い"
print("重みファイル:", sorted(f for f in os.listdir(WEIGHTS) if f.endswith(".pt")))


In [ ]:
# ---- 前処理: 学習時からの逐語コピー -----------------------------------------
# 学習時の前処理とズレると、スコアが静かに落ちる。書き直さずコピーする。

def pick_series(series_df):
    """study × 平面ごとに1本選ぶ。fluid-sensitive を優先。"""
    df = series_df.copy()
    df["_pref"] = -df["Fluid_Sensitive"].fillna(0).astype(int)
    return (df.sort_values(["StudyInstanceUID", "Anatomical_Plane",
                            "_pref", "SeriesInstanceUID"])
              .groupby(["StudyInstanceUID", "Anatomical_Plane"], as_index=False)
              .head(1))


def slice_position(ds):
    """IOP の法線への射影 = 真のスライス位置。

    z成分をそのまま使ってはならない。Sagittal ではスライス軸は X、
    Coronal では Y になる。
    """
    iop = getattr(ds, "ImageOrientationPatient", None)
    ipp = getattr(ds, "ImagePositionPatient", None)
    if iop is None or ipp is None:
        return None
    n = np.cross([float(v) for v in iop[:3]], [float(v) for v in iop[3:]])
    return float(np.dot([float(v) for v in ipp], n))


def load_volume(split, study, series):
    d = os.path.join(INPUT, f"{split}_series", study, series)
    try:
        paths = [f.path for f in os.scandir(d) if f.name.endswith(".dcm")]
    except OSError as e:
        return None, f"scandir: {e}"
    if not paths:
        return None, "no dcm"

    items = []
    for p in paths:
        try:
            ds = pydicom.dcmread(p, force=True)
            arr = ds.pixel_array
        except Exception:
            continue
        ps = getattr(ds, "PixelSpacing", None)
        items.append((slice_position(ds), arr,
                      float(ps[0]) if ps is not None else None))
    if not items:
        return None, "all slices failed to decode"

    if all(it[0] is None for it in items):
        items = [(i, a, s) for i, (_, a, s) in enumerate(items)]
    else:
        items = [it for it in items if it[0] is not None]
        items.sort(key=lambda x: x[0])

    out = np.zeros((N_SLICES, HW, HW), dtype=np.uint8)
    idx = np.linspace(0, len(items) - 1, N_SLICES).round().astype(int)
    for k, i in enumerate(idx):
        _, a, spacing = items[i]
        a = a.astype(np.float32)
        if a.ndim != 2:
            a = a[..., 0] if a.ndim == 3 else a.reshape(a.shape[-2:])
        if spacing and spacing > 0:
            # 物理的な視野を切り出してからリサイズする(§5)
            crop_px = int(round(FOV_MM / spacing))
            crop_px = max(16, min(crop_px, min(a.shape)))
            cy, cx = a.shape[0] // 2, a.shape[1] // 2
            h = crop_px // 2
            a = a[max(0, cy - h):cy + h, max(0, cx - h):cx + h]
        flat = a[::4, ::4].ravel()
        lo, hi = np.percentile(flat, [1, 99]) if flat.size else (0.0, 1.0)
        a = np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)
        out[k] = (cv2.resize(a, (HW, HW),
                             interpolation=cv2.INTER_AREA) * 255).astype(np.uint8)
    return out, None


def build_study(args):
    study, chosen = args
    vol = np.zeros((PLANES, N_SLICES, HW, HW), dtype=np.uint8)
    for pi, plane in enumerate(PLANES_ORDER):
        sid = chosen.get(plane)
        if sid is None:
            continue
        v, _ = load_volume("test", study, sid)
        if v is not None:
            vol[pi] = v
    return study, vol


In [ ]:
# ---- モデル: 学習時と字句レベルで同一でなければならない ----------------------
# 食い違うと state_dict のキーか形状がずれ、推論が静かに壊れる。

PRETRAINED = False       # backbone の重みはチェックポイントに入っている
USE_TRIPLET_POS = False


def build_encoder(backbone):
    kw = dict(pretrained=PRETRAINED, num_classes=0, in_chans=3)
    if backbone.startswith("vit_"):
        # DINOv2 は patch14 / 事前学習は518px。img_size を渡すと
        # timm が位置埋め込みを再サンプルして読み込む。
        assert HW % 14 == 0, f"patch14 の ViT には HW が14の倍数である必要がある: {HW}"
        kw["img_size"] = HW
    return timm.create_model(backbone, **kw)


class Net(nn.Module):
    """2.5D エンコーダ + 所見ごとの triplet attention。

    attention は (平面, 組, 所見) ごとに重みを出す。半月板を読むのに良い組と
    ACL を読むのに良い組は違うので、ここを所見ごとにするのが要点になる。
    1ベクトルに潰してから12所見へ分岐すると、この区別は取り戻せない。
    """

    def __init__(self, backbone, n_out=len(TARGETS)):
        super().__init__()
        self.enc = build_encoder(backbone)
        d = self.enc.num_features
        self.n_out = n_out
        self.att = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, n_out))
        self.pos = nn.Parameter(torch.zeros(1, 1, TRIPLETS, d))
        self.drop = nn.Dropout(0.2)
        self.cls_w = nn.Parameter(torch.empty(n_out, PLANES * d))
        self.cls_b = nn.Parameter(torch.zeros(n_out))
        nn.init.normal_(self.cls_w, std=0.02)

    def forward(self, x):                      # (B, 3, 24, H, W)
        B = x.shape[0]
        x = x.reshape(B * PLANES * TRIPLETS, 3, HW, HW)
        f = self.enc(x).reshape(B, PLANES, TRIPLETS, -1)
        if USE_TRIPLET_POS:
            f = f + self.pos
        w = torch.softmax(self.att(f), dim=2)          # 組の方向に正規化
        p = torch.einsum("bptd,bptc->bpcd", f, w)      # 所見ごとの平面埋め込み
        p = self.drop(p.permute(0, 2, 1, 3).reshape(B, self.n_out, -1))
        return (p * self.cls_w.unsqueeze(0)).sum(-1) + self.cls_b


nets = []
for f in range(5):
    p = os.path.join(WEIGHTS, f"model_fold{f}.pt")
    if not os.path.exists(p):
        continue
    m = Net(BACKBONE).to(dev).eval()
    m.load_state_dict(torch.load(p, map_location=dev))
    nets.append(m)
assert nets, f"重みが1つも読めなかった ({WEIGHTS})"
print(f"{len(nets)} fold 読み込み: {BACKBONE}")


In [ ]:
# ---- 推論して submission.csv を書く -----------------------------------------
t0 = time.time()

test = pd.read_csv(f"{INPUT}/test.csv")
te_series = pd.read_csv(f"{INPUT}/test_series.csv")
print(f"test: {len(test):,} 件の検査 / {len(te_series):,} シリーズ")

sel = pick_series(te_series)
tasks = [(s, dict(zip(g.Anatomical_Plane, g.SeriesInstanceUID)))
         for s, g in sel.groupby("StudyInstanceUID")]
print(f"{len(tasks):,} 件に対し {len(sel):,} シリーズを選択")

order, chunks = [], []
workers = max(1, (os.cpu_count() or 4))
with ProcessPoolExecutor(max_workers=workers) as ex:
    buf_u, buf_x = [], []
    for i, (study, vol) in enumerate(ex.map(build_study, tasks, chunksize=4), 1):
        buf_u.append(study)
        buf_x.append(vol)
        last = (i == len(tasks))
        if len(buf_u) < BATCH and not last:
            continue

        x = torch.from_numpy(np.stack(buf_x)).float().div_(255.0)
        x = ((x - 0.449) / 0.226).to(dev)
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=(dev == "cuda")):
            # 5 fold を確率平均する(学習時の合成手続きと同じ)
            p = sum(torch.sigmoid(m(x)) for m in nets) / len(nets)
        chunks.append(p.float().cpu().numpy())
        order += buf_u
        buf_u, buf_x = [], []

        if i % 200 == 0 or last:
            el = time.time() - t0
            print(f"    {i:,}/{len(tasks):,}  {el/60:.1f} min  "
                  f"残り {(len(tasks)-i)*el/max(i,1)/60:.0f} min", flush=True)
        if time.time() - t0 > TIME_BUDGET:
            print("[budget] 時間切れ。残りは0.5で埋める")
            break

P = np.concatenate(chunks) if chunks else np.zeros((0, len(TARGETS)))
n = len(order)
assert len(P) == n, f"{len(P)} != {n}"

# AUC は順位しか見ないので、列ごとに順位へ直してから書き出す。
# 単一モデルなら単調変換なのでスコアは変わらないが、複数モデルを混ぜるときと
# 同じ手続きにしておく(ここがずれると「検証した混合」と「提出した混合」が別物になる)。
pred = np.column_stack([rankdata(P[:, j]) / max(n, 1) for j in range(len(TARGETS))])
preds = dict(zip(order, pred))

sub = pd.DataFrame({"StudyInstanceUID": test.StudyInstanceUID})
arr = np.full((len(sub), len(TARGETS)), 0.5, np.float64)
for i, u in enumerate(sub.StudyInstanceUID):
    if u in preds:
        arr[i] = preds[u]
for j, t in enumerate(TARGETS):
    sub[t] = arr[:, j]

sample = pd.read_csv(f"{INPUT}/sample_submission.csv")
sub = sub[list(sample.columns)]
assert list(sub.columns) == list(sample.columns)
assert len(sub) == len(test)
assert sub[TARGETS].notna().all().all()
sub.to_csv(f"{OUT}/submission.csv", index=False)

n_pred = sum(1 for u in sub.StudyInstanceUID if u in preds)
print(f"\nsubmission.csv を書き出し {sub.shape}  予測済み {n_pred:,}/{len(sub):,} "
      f"(0.5で埋めたのは {len(sub)-n_pred:,})")
print(f"done in {(time.time()-t0)/60:.1f} min")
